# 🎯 Word-Level Training — 48 Videos (v4)
**Data:** 48 EMNLP videos with WavLM+prosody word-level features
**Download:** Downloads dataset from Kaggle explicitly
**GPU:** Tesla P100


In [ ]:
# Cell 1: Download dataset from Kaggle
import os
import subprocess

DATA_DIR = '/kaggle/input/chucklenet-48v-wordlevel'

# Check if already downloaded
if os.path.exists(DATA_DIR):
    files = os.listdir(DATA_DIR)
    print(f'Dataset already exists: {len(files)} items')
    print(f'Items: {files[:5]}')
else:
    print('Downloading dataset...')
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'subhajitdas/chucklenet-48v-wordlevel',
        '-p', '/kaggle/input/',
        '--unzip'
    ], capture_output=True, text=True)
    print('STDOUT:', result.stdout[:500])
    print('STDERR:', result.stderr[:500])
    
    # List what we got
    if os.path.exists(DATA_DIR):
        print(f'Downloaded: {os.listdir(DATA_DIR)}')
    else:
        # Try alternate path
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files:
                if f.endswith('.npy'):
                    print(f'Found: {os.path.join(root, f)}')
                    break

print('Setup complete')

In [ ]:
# Cell 2: Load data
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Find the data directory
import os
BASE_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('_features.npy') for f in files):
        BASE_DIR = root
        break

if BASE_DIR is None:
    raise FileNotFoundError('Could not find dataset with _features.npy files')

print(f'Data base dir: {BASE_DIR}')

# Find features and labels subdirs
FEAT_DIR = os.path.join(BASE_DIR, 'features') if os.path.exists(os.path.join(BASE_DIR, 'features')) else BASE_DIR
LABEL_DIR = os.path.join(BASE_DIR, 'labels') if os.path.exists(os.path.join(BASE_DIR, 'labels')) else BASE_DIR

print(f'Feature dir: {FEAT_DIR}')
print(f'Label dir: {LABEL_DIR}')

# Load all features
X_list, y_list, vids_list = [], [], []
for f in sorted(os.listdir(FEAT_DIR)):
    if not f.endswith('_features.npy'): continue
    vid = f.replace('_features.npy', '')
    feat_path = f'{FEAT_DIR}/{f}'
    label_path = f'{LABEL_DIR}/{vid}_labels.npy'
    if not os.path.exists(label_path):
        label_path = f'{LABEL_DIR}/{vid}_labels.npy'
    if not os.path.exists(label_path): continue
    
    X = np.load(feat_path)
    y = np.load(label_path)
    if len(X) != len(y): X = X[:len(y)]
    X_list.append(X)
    y_list.append(y)
    vids_list.extend([vid] * len(y))

X_all = np.vstack(X_list)
y_all = np.concatenate(y_list)
vids_all = np.array(vids_list)
X_all = np.nan_to_num(X_all.astype(np.float32), nan=0.0)

unique_vids = sorted(set(vids_list))
pos_rate = y_all.mean()

print(f'Data: {len(unique_vids)} videos, {len(y_all)} words')
print(f'Positive rate: {100*pos_rate:.1f}%')
print(f'Features: {X_all.shape}')

In [ ]:
# Cell 3: Model + Training
class WordLevelMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

gkf = GroupKFold(n_splits=min(5, len(unique_vids)))
pos_weight = min((1-pos_rate)/max(pos_rate, 1e-6), 3.0)
print(f'pos_weight: {pos_weight:.2f}')

fold_f1s = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, vids_all)):
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]
    if yte.sum()==0 or ytr.sum()==0: continue
    
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr).astype(np.float32)
    Xte_s = scaler.transform(Xte).astype(np.float32)
    
    model = WordLevelMLP(input_dim=X_all.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    
    Xtr_t = torch.tensor(Xtr_s).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)
    
    best_f1, patience, no_imp = 0, 10, 0
    for epoch in range(60):
        model.train()
        perm = torch.randperm(len(Xtr_t)).to(device)
        for i in range(0, len(Xtr_t), 128):
            idx = perm[i:i+128]
            if len(idx) < 2: continue
            opt.zero_grad()
            out = model(Xtr_t[idx])
            weights = torch.where(ytr_t[idx]==1, pos_weight, 1.0).to(device)
            loss = -(weights * (ytr_t[idx]*torch.log(out+1e-8) + (1-ytr_t[idx])*torch.log(1-out+1e-8))).mean()
            loss.backward(); opt.step()
        
        model.eval()
        with torch.no_grad():
            probs = model(torch.tensor(Xte_s).to(device)).squeeze().cpu().numpy()
        f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
        if f > best_f1: best_f1 = f; no_imp = 0
        else: no_imp += 1
        if no_imp >= patience: break
    
    model.eval()
    with torch.no_grad():
        probs = model(torch.tensor(Xte_s).to(device)).squeeze().cpu().numpy()
    p = precision_score(yte, (probs>=0.5).astype(int), zero_division=0)
    r = recall_score(yte, (probs>=0.5).astype(int), zero_division=0)
    f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
    fold_f1s.append(f)
    print(f'Fold {fold+1}: F1={f:.4f} P={p:.4f} R={r:.4f}')

print(f'\nCV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}')

In [ ]:
# Cell 4: Save results
import json
torch.save(model.state_dict(), '/kaggle/working/word_level_model.pt')
results = {
    'n_videos': len(unique_vids),
    'n_words': int(len(y_all)),
    'positive_rate': float(pos_rate),
    'cv_f1': float(np.mean(fold_f1s)),
    'cv_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s]
}
with open('/kaggle/working/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: word_level_model.pt, results.json')
print(json.dumps(results, indent=2))